## video inspection frame extraction and annotation preparation

- Imports required packages: pathlib, cv2, pandas, and shutil.
- Defines the main project directory and relevant subfolders.
- Defines paths to the 1-minute and 15-minute input videos.
- Selects one video for processing.
- Checks whether the selected video exists.
- Reads basic video information such as FPS, frame count, resolution, and duration.
- Creates project folders for data, extracted frames, metadata, YOLO data, scripts, notebooks, and runs.
- Defines a function to extract frames from the selected video.
- Runs a small test extraction with 20 frames.
- Extracts one frame per second from the selected video.
- Saves extracted frames as .jpg files.
- Creates a metadata table for the extracted frames.
- Saves the metadata table as a .csv file.
- Creates folders for manual annotation.
- Copies extracted frames into an annotation image folder.
- Prepares the data for manual bounding-box annotation in an external annotation tool.

### Imports and Paths

In [1]:
from pathlib import Path
import cv2
import pandas as pd
import shutil

In [2]:
# Project directory
# "." means: the folder where your notebook is located
PROJECT_DIR = Path(".")

# Input video folder
VIDEO_DIR = PROJECT_DIR / "videos"

# Your only video
VIDEO_PATH = VIDEO_DIR / "20250813_Group6_0650-0750_15min.mp4"

# Output folders
DATA_DIR = PROJECT_DIR / "data"
FRAME_DIR = DATA_DIR / "extracted_frames"
ANNOTATION_DIR = DATA_DIR / "annotation_data"
METADATA_DIR = DATA_DIR / "metadata"

# Three parts of the video
PART_FOLDERS = [
    "part_01_first_5min",
    "part_02_middle_5min",
    "part_03_last_5min",
]

# Create main folders
DATA_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Create three empty annotation folders
for part in PART_FOLDERS:
    (ANNOTATION_DIR / part).mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("Video path:", VIDEO_PATH.resolve())
print("Video exists:", VIDEO_PATH.exists())

Project directory: /mnt/c/Users/linap/OneDrive/Documents/Master/SoSe2026/deepLearning/code
Video path: /mnt/c/Users/linap/OneDrive/Documents/Master/SoSe2026/deepLearning/code/videos/20250813_Group6_0650-0750_15min.mp4
Video exists: True


### Video inspection

In [3]:
def inspect_video(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_sec = frame_count / fps if fps > 0 else None

    cap.release()

    info = {
        "video_path": str(video_path),
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": duration_sec,
        "duration_min": duration_sec / 60 if duration_sec else None,
    }

    return info

In [4]:
video_info = inspect_video(VIDEO_PATH)
video_info

{'video_path': 'videos/20250813_Group6_0650-0750_15min.mp4',
 'fps': 49.95,
 'frame_count': 44954,
 'width': 720,
 'height': 576,
 'duration_sec': 899.9799799799799,
 'duration_min': 14.999666332999665}

### Extract frames from video (three parts with 80 frames each)

In [5]:
def extract_frames_in_three_parts(
    video_path: Path,
    frame_base_dir: Path,
    part_folders: list[str],
    frames_per_part: int = 80,
    image_quality: int = 90,
    clear_existing_frames: bool = True
):
    """
    Extract frames from one video into three temporal parts.
    
    The video is split into three equal time sections.
    From each section, exactly frames_per_part frames are extracted
    in chronological order.
    """

    if not video_path.exists():
        raise FileNotFoundError(f"Video file does not exist: {video_path}")

    if clear_existing_frames and frame_base_dir.exists():
        shutil.rmtree(frame_base_dir)

    frame_base_dir.mkdir(parents=True, exist_ok=True)

    for part in part_folders:
        (frame_base_dir / part).mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        cap.release()
        raise ValueError("Could not read FPS from video.")

    if frame_count <= 0:
        cap.release()
        raise ValueError("Could not read frame count from video.")

    video_stem = video_path.stem
    total_saved = 0

    print(f"Video: {video_path.name}")
    print(f"FPS: {fps}")
    print(f"Total frames: {frame_count}")
    print(f"Duration: {frame_count / fps / 60:.2f} minutes")
    print()

    for part_idx, part_name in enumerate(part_folders):
        part_output_dir = frame_base_dir / part_name

        start_frame = int(round(part_idx * frame_count / 3))
        end_frame = int(round((part_idx + 1) * frame_count / 3)) - 1

        target_frame_indices = [
            int(round(start_frame + i * (end_frame - start_frame) / (frames_per_part - 1)))
            for i in range(frames_per_part)
        ]

        saved_in_part = 0

        for local_idx, frame_number in enumerate(target_frame_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
            ret, frame = cap.read()

            if not ret:
                print(f"Warning: Could not read frame {frame_number}. Skipping.")
                continue

            time_sec = frame_number / fps
            time_min = time_sec / 60

            filename = (
                f"{video_stem}_"
                f"{part_name}_"
                f"local_{local_idx:03d}_"
                f"frame_{frame_number:06d}_"
                f"time_{time_sec:07.2f}.jpg"
            )

            output_path = part_output_dir / filename

            cv2.imwrite(
                str(output_path),
                frame,
                [cv2.IMWRITE_JPEG_QUALITY, image_quality]
            )

            saved_in_part += 1
            total_saved += 1

        print(f"{part_name}: saved {saved_in_part} frames")
        print(f"Approx. time range: {start_frame / fps / 60:.2f} - {end_frame / fps / 60:.2f} min")
        print()

    cap.release()

    print(f"Total saved frames: {total_saved}")
    print(f"Saved to: {frame_base_dir}")

### Extract 240 frames from the video

In [6]:
extract_frames_in_three_parts(
    video_path=VIDEO_PATH,
    frame_base_dir=FRAME_DIR,
    part_folders=PART_FOLDERS,
    frames_per_part=80,
    image_quality=90,
    clear_existing_frames=True
)

Video: 20250813_Group6_0650-0750_15min.mp4
FPS: 49.95
Total frames: 44954
Duration: 15.00 minutes

part_01_first_5min: saved 80 frames
Approx. time range: 0.00 - 5.00 min

part_02_middle_5min: saved 80 frames
Approx. time range: 5.00 - 10.00 min

part_03_last_5min: saved 80 frames
Approx. time range: 10.00 - 15.00 min

Total saved frames: 240
Saved to: data/extracted_frames


### Create frame metadata table
saves: image name, image path, source video, frame number, time in seconds & time in minutes

In [7]:
def create_frame_metadata_from_parts(
    frame_base_dir: Path,
    output_csv: Path
):
    image_paths = sorted(frame_base_dir.glob("*/*.jpg"))

    records = []

    for img_path in image_paths:
        name = img_path.stem
        part = img_path.parent.name

        source_video = None
        local_index = None
        frame_number = None
        time_sec = None

        if "_part_" in name:
            source_video = name.split("_part_")[0]

        try:
            if "_local_" in name:
                local_index = int(name.split("_local_")[1].split("_frame_")[0])

            if "_frame_" in name:
                frame_number = int(name.split("_frame_")[1].split("_time_")[0])

            if "_time_" in name:
                time_sec = float(name.split("_time_")[1])
        except Exception:
            pass

        records.append({
            "part": part,
            "image_name": img_path.name,
            "image_path": str(img_path),
            "source_video": source_video,
            "local_index": local_index,
            "frame_number": frame_number,
            "time_sec": time_sec,
            "time_min": time_sec / 60 if time_sec is not None else None,
        })

    df = pd.DataFrame(records)

    part_order = {
        "part_01_first_5min": 1,
        "part_02_middle_5min": 2,
        "part_03_last_5min": 3,
    }

    df["part_order"] = df["part"].map(part_order)
    df = df.sort_values(["part_order", "frame_number"]).drop(columns=["part_order"])

    df.to_csv(output_csv, index=False)

    print(f"Metadata table saved to: {output_csv}")
    print(f"Number of frames: {len(df)}")

    return df

### Metadata for the video

In [8]:
METADATA_CSV = METADATA_DIR / f"frame_metadata_{VIDEO_PATH.stem}_240frames.csv"

frame_metadata = create_frame_metadata_from_parts(
    frame_base_dir=FRAME_DIR,
    output_csv=METADATA_CSV
)

frame_metadata.head()

Metadata table saved to: data/metadata/frame_metadata_20250813_Group6_0650-0750_15min_240frames.csv
Number of frames: 240


,part,image_name,image_path,source_video,local_index,frame_number,time_sec,time_min
0,part_01_first_5min,20250813_Group6_0650-0750_15min_part_01_first_...,data/extracted_frames/part_01_first_5min/20250...,20250813_Group6_0650-0750_15min,0,0,0.00,0.000000
1,part_01_first_5min,20250813_Group6_0650-0750_15min_part_01_first_...,data/extracted_frames/part_01_first_5min/20250...,20250813_Group6_0650-0750_15min,1,190,3.80,0.063333
2,part_01_first_5min,20250813_Group6_0650-0750_15min_part_01_first_...,data/extracted_frames/part_01_first_5min/20250...,20250813_Group6_0650-0750_15min,2,379,7.59,0.126500
3,part_01_first_5min,20250813_Group6_0650-0750_15min_part_01_first_...,data/extracted_frames/part_01_first_5min/20250...,20250813_Group6_0650-0750_15min,3,569,11.39,0.189833
4,part_01_first_5min,20250813_Group6_0650-0750_15min_part_01_first_...,data/extracted_frames/part_01_first_5min/20250...,20250813_Group6_0650-0750_15min,4,759,15.20,0.253333


In [9]:
frame_metadata["part"].value_counts()

part
part_01_first_5min     80
part_02_middle_5min    80
part_03_last_5min      80
Name: count, dtype: int64

### What to do next:

find the images for annotation here:
data/annotation_data/images/

Upload these images to an annotation tool, such as:
CVAT, Label Studio, Roboflow, makesense.ai, or LabelImg (easiest:https://www.makesense.ai/)

To start, label only one class: dog

Then export the labels in YOLO format.

Save the exported .txt files here: data/annotation_data/labels/